<a href="https://colab.research.google.com/github/Thanjaivalavan/M2-GenAI-AgenticAI/blob/main/LLM_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

# Set seed for reproducibility
torch.manual_seed(42)

# ==========================================
# STEP: PROMPTING & TOKENIZATION
# ==========================================
model_name = "distilbert/distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

prompt = "Agentic AI is"
inputs = tokenizer(prompt, return_tensors="pt")
token_ids = inputs["input_ids"]

print("1: Tokenization")
print(f"Raw Prompt   : '{prompt}'")
print(f"Token Tokens : {[tokenizer.decode([t]) for t in token_ids[0]]}")
print(f"Token IDs    : {token_ids[0].tolist()}\n")

# ==========================================
# STEP: EMBEDDINGS & TRANSFORMER FORWARD PASS
# ==========================================
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits  # Shape: [batch_size, sequence_length, vocab_size]

# Extract the logits for the very last predicted token position
next_token_logits = logits[0, -1, :]  # Dimension: [50257]

print("2: Vocab Logits")
print(f"Logits Tensor Shape : {next_token_logits.shape}")
print(f"Sample Logits       : {next_token_logits[:5].tolist()}\n")

# ==========================================
# STEP: SOFTMAX & TEMPERATURE SCALING
# ==========================================
def inspect_probabilities(logits, temperature=1.0):
    scaled_logits = logits / temperature
    probs = F.softmax(scaled_logits, dim=-1)
    return probs

probs_default = inspect_probabilities(next_token_logits, temperature=1.0)
probs_low_temp = inspect_probabilities(next_token_logits, temperature=0.2)
probs_high_temp = inspect_probabilities(next_token_logits, temperature=1.5)

top_probs, top_indices = torch.topk(probs_default, k=5)

print("3: Probabilities & Temperature")
print("Top 5 Predictions (T=1.0):")
for prob, idx in zip(top_probs, top_indices):
    token_str = tokenizer.decode([idx.item()])
    print(f"  Token: '{token_str}' | Probability: {prob.item():.4f}")

print(f"\nMax Token Probability at T=0.2 (Sharper) : {probs_low_temp.max().item():.4f}")
print(f"Max Token Probability at T=1.5 (Flatter) : {probs_high_temp.max().item():.4f}\n")

# ==========================================
# STEP: DECODING & GENERATION LOOP
# ==========================================
print("4: Autoregressive Loop")
generated_ids = token_ids.clone()
max_new_tokens = 5

for step in range(max_new_tokens):
    with torch.no_grad():
        out = model(generated_ids)
        next_logits = out.logits[0, -1, :]

        # Greedy decoding: pick highest probability token
        next_id = torch.argmax(next_logits, dim=-1).unsqueeze(0).unsqueeze(0)
        generated_ids = torch.cat([generated_ids, next_id], dim=-1)

        print(f"Step {step+1}: Appended '{tokenizer.decode(next_id[0])}' ➔ Full Text: '{tokenizer.decode(generated_ids[0])}'")

1: Tokenization
Raw Prompt   : 'Agentic AI is'
Token Tokens : ['Agent', 'ic', ' AI', ' is']
Token IDs    : [36772, 291, 9552, 318]



Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

2: Vocab Logits
Logits Tensor Shape : torch.Size([50257])
Sample Logits       : [-60.63502883911133, -62.00029373168945, -64.58641815185547, -65.14569854736328, -63.49659729003906]

3: Probabilities & Temperature
Top 5 Predictions (T=1.0):
  Token: ' a' | Probability: 0.2780
  Token: ' the' | Probability: 0.0868
  Token: ' an' | Probability: 0.0716
  Token: ' not' | Probability: 0.0395
  Token: ' one' | Probability: 0.0194

Max Token Probability at T=0.2 (Sharper) : 0.9959
Max Token Probability at T=1.5 (Flatter) : 0.0429

4: Autoregressive Loop
Step 1: Appended ' a' ➔ Full Text: 'Agentic AI is a'
Step 2: Appended ' new' ➔ Full Text: 'Agentic AI is a new'
Step 3: Appended ' type' ➔ Full Text: 'Agentic AI is a new type'
Step 4: Appended ' of' ➔ Full Text: 'Agentic AI is a new type of'
Step 5: Appended ' AI' ➔ Full Text: 'Agentic AI is a new type of AI'
